# Statistical Robustness and Advanced Methods

This notebook demonstrates advanced statistical methods for robust A/B testing analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Import our enhanced ab-glm package
from ab_glm import (
    simulate_ab_data,
    fit_binomial_glm,
    marginal_effects_ate_and_rr
)
from ab_glm.statistical_tests import (
    bootstrap_ci,
    permutation_test,
    power_analysis,
    sample_ratio_mismatch_test,
    heterogeneous_treatment_effects,
    variance_reduction_cuped,
    bayesian_ab_test
)
from ab_glm.causal_inference import (
    double_ml_ate,
    propensity_score_matching,
    check_covariate_balance
)
from ab_glm.robustness_checks import (
    complete_robustness_analysis,
    sensitivity_to_specification,
    meta_analysis
)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

## 1. Generate Sample Data with Complex Patterns

In [ ]:
# Generate data with heterogeneous treatment effects
n = 2000

# Create covariates
data = pd.DataFrame({
    'age': np.random.normal(35, 10, n),
    'income': np.random.lognormal(10.5, 0.7, n),
    'education': np.random.choice(['high_school', 'college', 'graduate'], n, p=[0.3, 0.5, 0.2]),
    'device': np.random.choice(['mobile', 'desktop'], n, p=[0.6, 0.4]),
    'country': np.random.choice(['US', 'UK', 'EU', 'Other'], n, p=[0.4, 0.2, 0.25, 0.15])
})

# Create treatment with some confounding
propensity = 1 / (1 + np.exp(-(
    -1 + 
    0.02 * (data['age'] - 35) + 
    0.3 * (data['education'] == 'graduate') -
    0.2 * (data['device'] == 'mobile')
)))
data['T'] = np.random.binomial(1, propensity)

# Create outcome with heterogeneous effects
base_rate = 0.1
treatment_effect = (
    0.05 +  # Base effect
    0.002 * (data['age'] - 35) +  # Age interaction
    0.03 * (data['education'] == 'graduate') +  # Education interaction
    0.02 * (data['device'] == 'desktop')  # Device interaction
)

outcome_prob = base_rate + data['T'] * treatment_effect
data['y'] = np.random.binomial(1, np.clip(outcome_prob, 0, 1))

# Add pre-treatment outcome for CUPED
data['y_pre'] = np.random.binomial(1, np.clip(base_rate + np.random.normal(0, 0.02, n), 0, 1))

print(f"Data shape: {data.shape}")
print(f"Treatment rate: {data['T'].mean():.2%}")
print(f"Outcome rate (control): {data[data['T']==0]['y'].mean():.2%}")
print(f"Outcome rate (treated): {data[data['T']==1]['y'].mean():.2%}")
print(f"Naive ATE: {data[data['T']==1]['y'].mean() - data[data['T']==0]['y'].mean():.4f}")

## 2. Bootstrap Confidence Intervals

In [ ]:
# Define ATE statistic function
def ate_statistic(df):
    treated = df[df['T'] == 1]['y'].mean()
    control = df[df['T'] == 0]['y'].mean()
    return treated - control

# Calculate bootstrap CI
print("Calculating bootstrap confidence intervals...")
boot_result = bootstrap_ci(
    data, 
    ate_statistic, 
    n_bootstrap=2000,
    confidence_level=0.95,
    random_state=42,
    show_progress=True
)

print(f"\nBootstrap Results:")
print(f"  ATE Estimate: {boot_result.estimate:.4f}")
print(f"  95% CI: [{boot_result.ci_lower:.4f}, {boot_result.ci_upper:.4f}]")
print(f"  Standard Error: {boot_result.std_error:.4f}")
print(f"  P-value: {boot_result.p_value:.4f}")

## 3. Permutation Test

In [ ]:
# Perform permutation test
perm_result = permutation_test(
    data, 'T', 'y',
    n_permutations=1000,
    two_sided=True,
    random_state=42,
    show_progress=True
)

print(f"\nPermutation Test Results:")
print(f"  Observed statistic: {perm_result.observed_statistic:.4f}")
print(f"  P-value: {perm_result.p_value:.4f}")
print(f"  Significant at α=0.05: {perm_result.p_value < 0.05}")

# Visualize null distribution
plt.figure(figsize=(10, 6))
plt.hist(perm_result.null_distribution, bins=50, alpha=0.7, edgecolor='black')
plt.axvline(perm_result.observed_statistic, color='red', linestyle='--', linewidth=2, label='Observed')
plt.axvline(0, color='black', linestyle='-', alpha=0.3)
plt.xlabel('Test Statistic (Difference in Means)')
plt.ylabel('Frequency')
plt.title('Permutation Test Null Distribution')
plt.legend()
plt.show()

## 4. Sample Ratio Mismatch Detection

In [ ]:
# Check for sample ratio mismatch
srm_result = sample_ratio_mismatch_test(
    n_treatment=data['T'].sum(),
    n_control=len(data) - data['T'].sum(),
    expected_ratio=0.5
)

print("Sample Ratio Mismatch Test:")
print(f"  Expected ratio: {srm_result['expected_ratio']:.2%}")
print(f"  Observed ratio: {srm_result['observed_ratio']:.2%}")
print(f"  P-value (binomial): {srm_result['p_value_binomial']:.4f}")
print(f"  P-value (chi-square): {srm_result['p_value_chi2']:.4f}")
print(f"  SRM detected: {srm_result['srm_detected']}")
if srm_result['warning']:
    print(f"  ⚠️ {srm_result['warning']}")

## 5. Power Analysis

In [ ]:
# Calculate statistical power for our experiment
observed_effect = boot_result.estimate
pooled_std = np.sqrt(
    data[data['T']==1]['y'].var() * 0.5 + 
    data[data['T']==0]['y'].var() * 0.5
)
cohen_d = observed_effect / pooled_std if pooled_std > 0 else 0

power_result = power_analysis(
    effect_size=cohen_d,
    sample_size=len(data) // 2,  # Per group
    alpha=0.05,
    alternative='two-sided'
)

print(f"Power Analysis:")
print(f"  Effect size (Cohen's d): {cohen_d:.3f}")
print(f"  Sample size per group: {len(data) // 2}")
print(f"  Statistical power: {power_result.power:.2%}")

# Calculate required sample size for 80% power
required_n = power_analysis(
    effect_size=cohen_d,
    power=0.8,
    alpha=0.05
)
print(f"  Required n for 80% power: {required_n.sample_size}")

## 6. Heterogeneous Treatment Effects

In [ ]:
# Analyze heterogeneous treatment effects
hte_results = heterogeneous_treatment_effects(
    data, 'T', 'y',
    subgroup_cols=['education', 'device', 'country'],
    min_samples=30
)

# Display results
print("Heterogeneous Treatment Effects:")
print("=" * 50)
for _, row in hte_results.iterrows():
    if row['significant']:
        sig_marker = "*"
    else:
        sig_marker = " "
    print(f"{row['subgroup_variable']:<15} = {row['subgroup_value']:<10}: "
          f"ATE = {row['effect']:+.4f} {sig_marker} (p={row['p_value']:.3f})")

# Visualize HTEs
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, var in enumerate(['education', 'device', 'country']):
    var_data = hte_results[hte_results['subgroup_variable'] == var]
    
    axes[idx].barh(var_data['subgroup_value'], var_data['effect'])
    axes[idx].axvline(0, color='black', linestyle='-', alpha=0.3)
    axes[idx].set_xlabel('Treatment Effect')
    axes[idx].set_title(f'HTE by {var.title()}')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Variance Reduction with CUPED

In [ ]:
# Apply CUPED variance reduction
cuped_result = variance_reduction_cuped(
    data, 'T', 'y', 'y_pre'
)

print("CUPED Variance Reduction:")
print(f"  Original ATE: {cuped_result['effect_original']:.4f} ± {cuped_result['se_original']:.4f}")
print(f"  CUPED ATE: {cuped_result['effect_cuped']:.4f} ± {cuped_result['se_cuped']:.4f}")
print(f"  Variance reduction: {cuped_result['variance_reduction']:.1%}")
print(f"  P-value (original): {cuped_result['p_value_original']:.4f}")
print(f"  P-value (CUPED): {cuped_result['p_value_cuped']:.4f}")
print(f"  Theta coefficient: {cuped_result['theta']:.3f}")

## 8. Bayesian A/B Testing

In [ ]:
# Perform Bayesian analysis
successes_a = data[data['T']==0]['y'].sum()
trials_a = len(data[data['T']==0])
successes_b = data[data['T']==1]['y'].sum()
trials_b = len(data[data['T']==1])

bayes_result = bayesian_ab_test(
    successes_a=successes_a,
    trials_a=trials_a,
    successes_b=successes_b,
    trials_b=trials_b,
    prior_alpha=1,
    prior_beta=1,
    n_simulations=100000
)

print("Bayesian A/B Test Results:")
print(f"  Probability B better than A: {bayes_result['prob_b_better']:.1%}")
print(f"  Expected lift: {bayes_result['expected_lift']:.2%}")
print(f"  95% CI for lift: [{bayes_result['lift_ci_lower']:.2%}, {bayes_result['lift_ci_upper']:.2%}]")
print(f"  Risk of choosing A: {bayes_result['risk_choosing_a']:.4f}")
print(f"  Risk of choosing B: {bayes_result['risk_choosing_b']:.4f}")
print(f"  Recommendation: {bayes_result['recommendation']}")

## 9. Causal Inference with Double ML

In [ ]:
# Prepare data for causal inference
X = pd.get_dummies(data[['age', 'income', 'education', 'device', 'country']], drop_first=True)
treatment = data['T'].values
outcome = data['y'].values

# Double ML estimation
print("Double/Debiased Machine Learning:")
dml_result = double_ml_ate(
    X, treatment, outcome,
    ml_method='rf',
    n_folds=5,
    n_estimators=100
)

print(f"  ATE: {dml_result['ate']:.4f} ± {dml_result['std_error']:.4f}")
print(f"  95% CI: [{dml_result['ci_lower']:.4f}, {dml_result['ci_upper']:.4f}]")
print(f"  T-statistic: {dml_result['t_statistic']:.3f}")
print(f"  P-value: {dml_result['p_value']:.4f}")

## 10. Propensity Score Matching

In [ ]:
# Propensity score matching
ps_result = propensity_score_matching(
    X, treatment, outcome,
    caliper=0.1,
    n_matches=1,
    with_replacement=False
)

print("Propensity Score Matching:")
print(f"  ATE: {ps_result.ate:.4f} ± {ps_result.std_error:.4f}")
print(f"  ATT: {ps_result.att:.4f}")
print(f"  Number of matched pairs: {len(ps_result.matched_pairs)}")
print(f"  Matching rate: {len(ps_result.matched_pairs) / treatment.sum():.1%}")

# Check balance after matching
print("\nCovariate Balance:")
balance = ps_result.balance_statistics
n_balanced = balance['balanced'].sum()
n_total = len(balance)
print(f"  Balanced covariates: {n_balanced}/{n_total} ({n_balanced/n_total:.1%})")

# Show unbalanced covariates
unbalanced = balance[~balance['balanced']]
if len(unbalanced) > 0:
    print("  Unbalanced covariates:")
    for _, row in unbalanced.iterrows():
        print(f"    - {row['variable']}: SMD = {row['std_mean_diff']:.3f}")

## 11. Complete Robustness Analysis

In [ ]:
# Prepare data in expected format for robustness analysis
df_robust = data.copy()
df_robust['user_id'] = range(len(df_robust))
df_robust['country_EU'] = (df_robust['country'] == 'EU').astype(int)
df_robust['device_mobile'] = (df_robust['device'] == 'mobile').astype(int)
df_robust['prior_views'] = np.random.poisson(5, len(df_robust))  # Simulated

# Run complete robustness analysis
robustness = complete_robustness_analysis(df_robust)

print("\n" + "="*60)
print("COMPLETE ROBUSTNESS ANALYSIS")
print("="*60)

# Baseline results
if 'baseline' in robustness and 'error' not in robustness['baseline']:
    print(f"\nBaseline Model:")
    print(f"  ATE: {robustness['baseline']['ate']:.4f}")
    print(f"  Risk Ratio: {robustness['baseline']['risk_ratio']:.4f}")

# Assumption checks
if 'assumptions' in robustness and 'warnings' in robustness['assumptions']:
    if robustness['assumptions']['warnings']:
        print(f"\nAssumption Warnings:")
        for warning in robustness['assumptions']['warnings']:
            print(f"  ⚠️ {warning}")

# Outlier robustness
if 'outlier_robustness' in robustness and 'error' not in robustness['outlier_robustness']:
    print(f"\nOutlier Robustness:")
    print(f"  Robust to outliers: {robustness['outlier_robustness']['robust']}")
    if 'sensitivity_range' in robustness['outlier_robustness']:
        range_val = robustness['outlier_robustness']['sensitivity_range']
        print(f"  ATE range: [{range_val[0]:.4f}, {range_val[1]:.4f}]")

# Specification sensitivity
if 'specification_sensitivity' in robustness and 'error' not in robustness['specification_sensitivity']:
    print(f"\nSpecification Sensitivity:")
    print(f"  Number of specifications tested: {robustness['specification_sensitivity']['n_specifications']}")
    if 'ate_range' in robustness['specification_sensitivity']:
        ate_range = robustness['specification_sensitivity']['ate_range']
        print(f"  ATE range across specs: [{ate_range[0]:.4f}, {ate_range[1]:.4f}]")

# Summary
print(f"\nSummary:")
print(f"  All checks passed: {robustness['summary']['all_checks_passed']}")
print(f"  Recommendation: {robustness['summary']['recommendation']}")

## 12. Meta-Analysis Example

In [ ]:
# Simulate multiple experiments for meta-analysis
experiments = []
n_experiments = 5

print("Simulating multiple experiments...")
for i in range(n_experiments):
    # Simulate an experiment
    n_exp = np.random.randint(500, 2000)
    effect = np.random.normal(0.05, 0.02)  # True effect with some variation
    se = np.sqrt(0.25 * (1/n_exp + 1/n_exp))  # Approximate SE
    
    experiments.append({
        'effect': effect + np.random.normal(0, se),  # Add sampling variation
        'se': se,
        'n': n_exp
    })
    print(f"  Exp {i+1}: n={n_exp}, effect={experiments[-1]['effect']:.4f}, se={se:.4f}")

# Perform meta-analysis
print("\nFixed Effects Meta-Analysis:")
meta_fixed = meta_analysis(experiments, method='fixed_effects')
print(f"  Pooled effect: {meta_fixed['pooled_effect']:.4f} ± {meta_fixed['pooled_se']:.4f}")
print(f"  P-value: {meta_fixed['p_value']:.4f}")
print(f"  I-squared: {meta_fixed['I_squared']:.1f}%")
print(f"  Heterogeneity p-value: {meta_fixed['p_heterogeneity']:.4f}")

print("\nRandom Effects Meta-Analysis:")
meta_random = meta_analysis(experiments, method='random_effects')
print(f"  Pooled effect: {meta_random['pooled_effect']:.4f} ± {meta_random['pooled_se']:.4f}")
print(f"  P-value: {meta_random['p_value']:.4f}")

# Visualize forest plot
fig, ax = plt.subplots(figsize=(10, 6))

# Individual studies
y_pos = np.arange(len(experiments))
effects = [exp['effect'] for exp in experiments]
ses = [exp['se'] for exp in experiments]
sizes = [exp['n'] for exp in experiments]

# Plot individual studies
for i, (eff, se, n) in enumerate(zip(effects, ses, sizes)):
    ax.errorbar(eff, i, xerr=1.96*se, fmt='o', markersize=np.sqrt(n/50), 
                capsize=5, label=f'Study {i+1}' if i < 3 else '')

# Plot pooled estimate
ax.errorbar(meta_fixed['pooled_effect'], len(experiments), 
            xerr=1.96*meta_fixed['pooled_se'], fmt='D', 
            markersize=10, color='red', capsize=5, label='Pooled (Fixed)')

ax.axvline(0, color='black', linestyle='-', alpha=0.3)
ax.axvline(meta_fixed['pooled_effect'], color='red', linestyle='--', alpha=0.3)
ax.set_yticks(range(len(experiments) + 1))
ax.set_yticklabels([f'Study {i+1}' for i in range(len(experiments))] + ['Pooled'])
ax.set_xlabel('Effect Size')
ax.set_title('Forest Plot: Meta-Analysis of Multiple Experiments')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated advanced statistical methods for robust A/B testing:

1. **Bootstrap CI**: Non-parametric confidence intervals
2. **Permutation Tests**: Distribution-free hypothesis testing
3. **Sample Ratio Mismatch**: Detection of randomization issues
4. **Power Analysis**: Sample size and power calculations
5. **Heterogeneous Treatment Effects**: Subgroup analysis
6. **CUPED**: Variance reduction using pre-treatment data
7. **Bayesian A/B Testing**: Probabilistic decision making
8. **Double ML**: Causal inference with machine learning
9. **Propensity Score Matching**: Quasi-experimental designs
10. **Robustness Analysis**: Comprehensive assumption checking
11. **Meta-Analysis**: Combining results from multiple experiments

These methods provide a comprehensive toolkit for ensuring reliable and robust A/B test results.